<a href="https://colab.research.google.com/github/dienerlab/2026_strain_course/blob/main/reconstruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://github.com/dienerlab/2026_strain_course/blob/main/docs/reconstruction/assets/mm_course_2026.png?raw=true)

# 🦠 Part 1: Metabolic reconstruction of microbial isolates 🧫

This notebook is the first chapter for the [2026 Course on Strain Metabolic Modeling](https://insights.medunigraz.at/en/events/detail/workshop-strain-level-metabolic-modeling-of-microbial-communities). The presentation slides can be [found here](https://dienerlab.github.io/2026_strain_course/reconstruction).

Save your own local copy of this notebook by using `File > Save a copy in Drive`. At some point you may be prompted to certify the notebook is safe. We promise that it is 🤞

**Disclaimer:**

The Google Colab notebook environment will interpret any command as Python code by default. If we want to run bash commands we will have to prefix them by `!`. So any command you see with a leading `!` is a bash command and if you wanted to run it in your terminal you would omit the leading `!`. For example, if in the Colab notebook you ran `!wget` you would just run `wget` in your terminal.

## Setup

The setup will make sure we have everything we need. First we will get the course materials.

Now we will install a small Python package that was built for the very purpose of this course and installing it will also pull in some additional commands and packages we need, for instance the [GAPSEQ](https://github.com/jotech/gapseq) tool and the metabolic modeling package [COBRAPY](https://opencobra.github.io/cobrapy) which we will use extensively.

In [ ]:
!pip uninstall -y strainy
!pip install -q git+https://github.com/dienerlab/2026_strain_course
!bootstrap

We can quickly check that `gapseq` was installed correctly by running its help command.

In [ ]:
!gapseq -h

That's it. We are ready to start.

# 🧬 From Genome to Model

## The Assembly
For our first step we will get our assembled genome. As noted before, these assemblies will not give us a singe contigious piece of DNA, but rather a set of fragmented *contigs*.

We will use the course Python package to get our own personal bacterial assembly. **Before running the code below, substitute `[YOUR NAME]` with your full name or nickname.** You can call yourself whatever you'd like, just make sure it's fairly unique. This name will be used to randomly assign you a specific genome assembly from the database.

What organism do you get?

In [ ]:
from strainy import gimme_genome

assembly = gimme_genome("Christian Diener")

This will download your assembled genome and tell you which organism you are dealing with. Let's take a peak at your assembly. What do you expect it to look like?

In [ ]:
!head *.fna

This is a [FASTA file](https://en.wikipedia.org/wiki/FASTA_format) with one entry for each contig. Not all assemblies are complete so you might see several contigs. We can count the contigs in our file.

In [ ]:
!seqkit stats *.fna

## The Reconstruction

With our genome sequence can start building the metabolic model. Let's recall the the basics of metabolic reconstruction and what we'll need.

![reconstruction methods](https://github.com/Gibbons-Lab/isb_course_2021/raw/main/docs/models/assets/reconstruction.png)

Roughly, we need to run the following steps:

1. identify enzymes and transporters in the sequence and map them to the universal model
2. find the model most consistent with the genomic evidence that can also produce biomass
3. curate the model with a growth medium

GAPSEQ can run all of these steps, so let's get started. There are some parameters we can set. For instance, we will use the diamond aligner which is much faster than the good ol' BLAST while giving us similar accuracy.

---

**🤔 What is this weird `${ASSEMBLY}` thing?**

*This is just a placeholder for the full assembly ID for your isolate. That way you don't have to type it every time. Nifty, isn't it?*

---

In [ ]:
!gapseq find -p all -A diamond ${ASSEMBLY}.fna
!gapseq find-transport -A diamond ${ASSEMBLY}.fna

Let's take a look what we got here. We essentially get 3 tables:

1. The identified reactions (`*-all-Reactions.tbl`)
2. The identified transporter (`*-Transporter.tbl`)
3. The found pathways (`*-all-Pathways.tbl`)

Feel free to inspect the first two a bit more in detail right now. What do you think those tell you? We will take a look at the pathways together.

In [ ]:
from os import environ
import pandas as pd

pws = pd.read_csv(environ["ASSEMBLY"] + "-all-Pathways.tbl", sep="\t", skiprows=3)
pws

This lists all the pathways in GAPSEQ and whether they were deemed present or not (`Prediction`). Normally GAPSEQ needs half of the reactions in a pathway to be present but his can be overwritten by the presence of "key reaction" which are important reactions/enzymes that usually indicate the presence of the full pathway.

We can also habe a look at all the detected pathways only:

In [ ]:
pws[pws.Prediction]

How many pathways were detected in you strain?

With the pathways and gram status we can start by creating a draft model. A draft model contains the following:

- all metabolic reactions within a present pathway
- a biomass reaction
- transport reactions based on the detected transporters

While it does contain a biomass formulation the model itself is probably no capable of producing biomass yet. It still contains **gaps**.

In [ ]:
!gapseq draft \
    -r ${ASSEMBLY}-all-Reactions.tbl \
    -t ${ASSEMBLY}-Transporter.tbl \
    -p ${ASSEMBLY}-all-Pathways.tbl

### Gapfilling

Before the model becomes useful we have to fill the gaps that block the production of biomass in the model. In general gapfilling means adding the least amount of additional reactions from the universal model that allow biomass production / growth. GAPSEQ is smart about this and prefers to add reactions that had some genomic evidence in your strain genome. For instance, by using enzymes that were not part of a complete pathway but had a good blast score or by including enzymes that had somewhat intermediate blast scores.

We could start right away by filling in our draft model as it is, but **why would that not be such a good idea?**

Well we don't really know what is present in the environment in the first place. We could assume that every metabolite with a detected transporter is there but that can be much too rich. If we know nothing, GAPSEQ can create a minimal medium for us based on the genomic evidence, but we know that our isolate came from the human gut, so we wil use a medium that represents this environment.

This will take a little bit (~30 minutes).

In [ ]:
from strainy import gimme_medium

medium = gimme_medium()
medium.to_csv("gut_medium_gapseq.csv", index=False)

medium

In [ ]:
!gapseq fill -m ${ASSEMBLY}-draft.RDS -n gut_medium_gapseq.csv -k 0.01

You will now find a new `*.xml` file in the directory that does *not* carry the draft designatio anymore. This is our final reconstruction at this point.

## The Model Check

We talked a bit about model curation and validation and learned that the best way to test a model is using real-world data. But, believe us, there are **plenty** of ways the model can be wrong even before you compare it to data.

This has been known in the metabolic modeling community for quite some time, and has been discussed extensively [here](https://www.embopress.org/doi/full/10.15252/msb.20156157) and [here](https://doi.org/10.1038/s41587-020-0446-y), for example.

Luckily, based on this prior feedback from the community, there is now a nice tool called [MEMOTE](https://memote.io), which allows you to upload an SBML model and run it through a whole battery of checks and tests.

In [ ]:
!memote report snapshot ${ASSEMBLY}.xml --filename ${ASSEMBLY}-memote.html

 To find the results, look through the files on the left (📁) and download the file ending in `.html`. You can open it with any web browser and each section has a detailed explanation of what was tested and why.

---

And with that we have finished building our model and with an environmentally aware gapfilling :tada:

**Any open questions?**

Cool, let's take a short break and then come back for part 2 where we will actually inspect the model a bunch more.

<br><br>

---

**Break time ☕  🥪**

---

<br><br>

# 💻 Part 2: From Model to Fluxes

Okay, now we have our metabolic reconstruction, so let's finally get to some actual metabolic modeling! We will make use of the [COBRAPY](https://opencobra.github.io/cobrapy/) package, which allows us to run a bunch of different analyses and simulations with our metabolic model.

COPRAPY is already installed in your notebook. If you need help with COBRAPY in the future, you can check out [its documentation](https://cobrapy.readthedocs.io/en/latest/).

## Metabolic models and COBRAPY

Before we actually start simulating the model let's have a look at it. We have learned that FBA builds on top of a lot of inferred information. So let's try to find this in the model we built. Here a short overview what ingredients we need to cook up an FBA result:

![flux balance analysis](https://github.com/Gibbons-Lab/isb_course_2021/raw/main/docs/models/assets/fba.png)

The first thing we have to to is to actually read the model into memory, and maybe inspect it a bit to see what is going on. **Before running the code cell, substitute `[YOUR ASSEMBLY]` with the ID of your assembly (including the `.xml` part).** You can find the file name ending in .xml on the left (📁).

In [ ]:
from os import environ
from cobra.io import read_sbml_model

model = read_sbml_model(environ["ASSEMBLY"] + ".xml")
model

Okay, this gives us some very basic info about the model, like the number of metabolites and reactions, and the model objective, which in our case is growth. The model has accessions for metabolites and reactions, which can be used to look up a whole bunch of information and annotations.

The objective is described, a bit opaquely, as growth. We talked a bit before about the biomass composition and we can have a look at this by taking a peak at the biomass reaction.

In [ ]:
summ = model.reactions.bio1.summary()
summ

This is nice but we can't really identify what the components are. Let's use a small hack to see the names of the metabolites instead:

In [ ]:
from IPython.display import HTML

HTML(summ.to_html(names=True))

This looks like what we'd expect. From the previous MEMOTE report, we know that the total consumed mass here should add up to something around 1gDW (± 0.05gDW). You can also see that the reactions are balanced. For instance, the consumed ATP is converted into ADP.

As we saw, the short names can be a bit cryptic.  Let's get some more info on, for instance, the 42nd metabolite in the model.

In [ ]:
model.metabolites[42]

Okay, this is much more informative. You get the full name of the metabolite, the molecular formula, and the reactions in the model that include this metabolite. The same kind of output can be generated for individual reactions in the model.

In [ ]:
model.reactions[42]

This gives us a lot of information about the reaction, including the substrates, products, reversibility, and bounds. The GPR is the *Gene-Protein-Reaction* rule and it states under what conditions the enzyme is functional.

If you want more information about a reaction you can also check out the annotations (see below).

In [ ]:
model.reactions[42].annotation

This will give you feature IDs for other databases, if available, which can be super-helpful when trying to map results back to experimental data. This is why the presence of sufficient annotations is a major criterion for MEMOTE scoring.

## Flux Balance Analysis

Okay, can we finally get to an actual simulation, please? 🥺 Yes! Running an FBA here is really simple. Just call `optimize()` on your model.

In [ ]:
fba_solution = model.optimize()
fba_solution

There is a lot of stuff going on under the hood here, so it's kind of cool that it runs so fast.

Here you see the predicted growth rate and the fluxes. We won't spend too much time on the reduced costs here, but they are metrics that can be helpful sometimes. The two things to know about reduced costs are:

1. They are only different from zero if the flux is zero.
2. They quantify by how much the growth rate would be reduced if we force a flux through this reaction.

So they are kind of the cost in growth rate the organism would have to pay to get flux through that previously shut-off reaction.

This output may be a bit complex. For any solution we can also get a summary output:

In [ ]:
summ = model.summary(fba_solution)

HTML(summ.to_html(names=True))

This is still fairly long, but definitely looks nicer. We now see what your bacterium eats (uptake) and what proportion of the total carbon influx comes from that particular uptake (C-Flux). We get the same output for secretion.

What is the major carbon source in for your organism in this simulation?

Does the organism secrete carbon-rich metabolites? If yes, which ones?


## 🌍 Comparative Reverse Ecology

Why are we interested in those uptake fluxes? Well, they are the recipe for an organism's growth, and thus, related to its evolutionary history (i.e. to the environmental conditions that it has adapted to live in). They are not solely what is present in the organism's surroundings, but the actual consumption preferences of that organism within a particular environment.

When we started constructing our models, we had a description of the (metabolic) environment (the available nutrients) and our organism's genome. But now, suddenly, we learn something about how our organism interacts with the environment. And that means we learned something about its **ecology** 🤯

Inferring information about an organism's ecology from genomic data is called [Reverse Ecology](https://en.wikipedia.org/wiki/Reverse_ecology). Check these [slides](https://gibbons-lab.github.io/isb_course_2021/models/#/26) for some more discussion on this point.

Before we get there, however, we need to curate our model a bit...

## Getting realistic: PFBA and media

As we noted, the particular flux solutions are often not unique and there may be many other solutions. To alleviate this let's use parsimonious FBA (pFBA), which minimizes the total import flux and the amount of enzymes required for growth.

![parsimonious FBA](https://github.com/dienerlab/2026_strain_course/raw/main/docs/reconstruction/assets/pfba.png)

pFBA is part of COBRAPY and will give us a new solution.

In [ ]:
from cobra.flux_analysis import pfba

pfba_solution = pfba(model)
summ = model.summary(pfba_solution)

HTML(summ.to_html(names=True))

We got our first real results -- awesome! But are they realistic? Remember that we stated that the biomass flux is essentially equivalent to the growth rate. We can convert the growth rate $\mu$ to the doubling time with the formula

$$
\tau = \ln(2) / \mu
$$

In [ ]:
from math import log

tau = log(2) / fba_solution.objective_value
print(f"The predicted dubling time is {tau:.2g} hours ({tau*60*60:.2g} seconds).")

## Metabolic flexibility

We have seen before that there are multiple flux solutions that can yield a speficic growth rate. Parsimonity is one principle to deal with this. However, we can also inspect the underlying variability. This can be done by flux varibaility analysis. Here the growth rate is fixed to some value close to the maximum (for instance 99%) and then each individual flux is minimized an d maximized thus giving the full range for each reaction while maintaining the growth rate.

In [ ]:
from cobra.flux_analysis import flux_variability_analysis

ranges = flux_variability_analysis(model, fraction_of_optimum=0.99, processes=2)

In [ ]:
ranges

Hard to see what is going on so let's add in the names of the reactions and the pFBA fluxes too.

In [ ]:
import pandas as pd

name_map = pd.Series({rxn.id: rxn.name for rxn in model.reactions})
ranges["name"] = name_map[ranges.index]
ranges["pfba_flux"] = pfba_solution.fluxes[ranges.index]

ranges

We can use that to plot the ranges vs the pfba flux.

In [ ]:
from plotnine import *

p = (
    ggplot(ranges) +
    aes(x="maximum - minimum") +
    geom_histogram(bins=20) +
    labs(x="flux range [max-min]")
)

p

So basically most fluxes are well-defined but some can vary a lot.

## Knockout simulations

We can also simulate what effect gene deletions would have on the growth of the organism. Let's then filter the genes that lowered the growrh rate to 0.01 or less.

In [ ]:
from cobra.flux_analysis import single_gene_deletion

deletions = single_gene_deletion(model)

bad_ones = deletions[deletions.growth < 0.01]
bad_ones

We could look at info for a gene.

In [ ]:
model.genes[0].annotation

So this could be used to figure out what gene is the underlying culprit.

# 🏠 Exercises

**1. Characterize essential genes**

Let's continue right away. We just got a list of genes that lower the growth rate. If they actually abolish growth (let's say μ<1e-6) we would call those essential genes. Use the annotations to look up some of them on UniRef. Do they make sense?

You can access individual gene annotations with `model.genes.[ID].annotation`, so for instance `model.genes.LR134375_1_977.annotation`.

In [ ]:
# Your code here

**2. "Uninformed media"**

When we talked about gapfilling we did mention that it can be hard to have a growth medium/environment for a given isolate. GAPSEQ can predict a minimal medium from the draft model. This can then be used for the gapfilling. Let's inspect this a little bit.

First we start by predicting the medium. Because it will likely use oxygen and we have anaerobes here, we will limit the prediction to anaerobic ones.

In [ ]:
!gapseq medium -m ${ASSEMBLY}-draft.RDS \
  -p ${ASSEMBLY}-all-Pathways.tbl \
  -o minimal_medium.csv \
  -c cpd00007:0

This gives us the following:

In [ ]:
import pandas as pd

pd.read_csv("minimal_medium.csv")

What do you notice here when comparing it with our gut medium?

To see how this affects the metabolic modeling will now also perform the gapfilling again and save this new model under a different name.

In [ ]:
!gapseq fill -m ${ASSEMBLY}-draft.RDS -n minimal_medium.csv -k 0.01 -f minimal_gapfill

You now can find this model in the `minimal_gapfill` folder.

Can you figure out what the major carbon sources are when simulating growth with FBA?

How does this compare to our "real" medium?

In [ ]:
# Your code here